# 03 - Train MLP from Pre-extracted Concerto Features

This notebook trains the translation head on the pre-extracted S3DIS Area 5 features. It is designed to run on a Google Colab T4 instance.

Ensure you have access to the shared Google Drive with the preprocessed data and extracted features.

### 1. Setup & Mount Drive

In [15]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Project'
DRIVE_FEATURES_DIR = f'{DRIVE_ROOT}/features/s3dis_area5'
LOCAL_FEATURES_ROOT = '/content/local_features'
LOCAL_FEATURES_DIR = f'{LOCAL_FEATURES_ROOT}/s3dis_area5'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

/content/Deep_learning_project
M	notebooks/pyproject.toml
Branch 'dev/enc-mlp' set up to track remote branch 'dev/enc-mlp' from 'origin'.
Reset branch 'dev/enc-mlp'
Your branch is up to date with 'origin/dev/enc-mlp'.
From https://github.com/Gandata/Deep_learning_project
 * branch            dev/enc-mlp -> FETCH_HEAD
Already up to date.
dev/enc-mlp
2905206 (HEAD -> dev/enc-mlp, origin/dev/enc-mlp) label path route changed and train


In [17]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto

fatal: destination path '/content/Concerto' already exists and is not an empty directory.


### 2. Symlink Data & Checkpoints
We map Drive data/checkpoints into the repository, but copy training features to local runtime storage so training is not bottlenecked by the Drive mount.

In [18]:
# Symlink Drive data/checkpoints and copy features locally for faster training
!rm -f data checkpoints features pretrained
!ln -sf {DRIVE_ROOT}/data ./data
!ln -sf {DRIVE_ROOT}/checkpoints ./checkpoints
!mkdir -p {LOCAL_FEATURES_ROOT}
!rm -rf {LOCAL_FEATURES_DIR}
!cp -r {DRIVE_FEATURES_DIR} {LOCAL_FEATURES_ROOT}/
!ln -sf {LOCAL_FEATURES_ROOT} ./features
!readlink -f ./features/s3dis_area5
!du -sh {LOCAL_FEATURES_DIR}

In [19]:
%cd notebooks/

/content/Deep_learning_project/notebooks


### 3. Setup Hugging Face Token
Input your Hugging Face token below if `open_clip_torch` needs it to download the models.

In [20]:
# If token is in colab secrets

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

### 4. Validate Inputs and Run Training
This section checks the extracted `.npz` files, fixes the config paths for Colab, and then launches MLP training.

In [21]:
# Forzar a uv a usar/instalar Python 3.10.12 y sincronizar
!uv python install 3.10.12
!uv sync --python 3.10.12

Python 3.10.12 is already installed
Resolved 147 packages in 0.82ms
Checked 141 packages in 1ms


In [22]:
!uv add dotenv
!uv add open_clip_torch

Resolved 147 packages in 0.79ms
Checked 141 packages in 1ms
Resolved 147 packages in 0.78ms
Checked 141 packages in 1ms


KeyboardInterrupt: 

In [24]:
import yaml
import os

# Define the file path
file_path = '/content/Deep_learning_project/configs/train_mlp_s3dis.yaml'
base_prefix = '/content/Deep_learning_project/'

# 1. Load the existing YAML content
with open(file_path, 'r') as f:
    config = yaml.safe_load(f)

# 2. Update the paths
# Update 'data' section
config['data']['train_features_path'] = os.path.join(base_prefix, config['data']['train_features_path'])
config['data']['label_embeddings_path'] = os.path.join(base_prefix, config['data']['label_embeddings_path'])

# Update 'training' section
config['training']['checkpoint_dir'] = os.path.join(base_prefix, config['training']['checkpoint_dir'])
config['training']['metrics_path'] = os.path.join(base_prefix, config['training']['metrics_path'])

# 3. Save the modified YAML back to the file
with open(file_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Successfully updated paths in {file_path}")

Successfully updated paths in /content/Deep_learning_project/configs/train_mlp_s3dis.yaml


In [ ]:
# Run the real extraction script with chunking to avoid T4 OOM on large rooms
#!PYTHONPATH=/content/Concerto uv run python /content/Deep_learning_project/scripts/extract_features.py --data_dir /content/Deep_learning_project/data/s3dis_processed --out_dir /content/Deep_learning_project/features/s3dis_area5 --feature-dtype float16 --max-points-per-chunk 100000
!PYTHONPATH=/content/Deep_learning_project:/content/Concerto uv run python /content/Deep_learning_project/src/train.py --config /content/Deep_learning_project/configs/train_mlp_s3dis.yaml


Using device: cuda
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


auto: notebook execution

# Other tests

In [ ]:
#!PYTHONPATH=/content/Concerto uv run python /content/Concerto/demo/0_pca.py

In [ ]:
#!PYTHONPATH=/content/Concerto uv run python -c "import sys, torch; sys.path.insert(0, '/content/Concerto'); import concerto; import spconv.pytorch as spconv; print('python ok'); print(torch.__version__, torch.version.cuda); print(spconv.__file__)"